In [18]:
# ============ p6 CONFIG ============
# Self-contained: install only if missing (rule-compliant: declared in-notebook)
!pip install lightgbm catboost scipy scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [19]:
import numpy as np, pandas as pd
from scipy import signal as sp_signal
from scipy.fft import rfft, rfftfreq
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings; warnings.filterwarnings("ignore")

FS = 50                 # target uniform rate (Hz)
SHORT_S, LONG_S, EXT_S = 1, 4, 6
N_CLASS = 9
SEED = 42
P5_BASELINE_BA = 0.91879   # <-- your verified p5 OOF BA; guard compares against this

# ---- PATHS (edit if yours differ) ----
PATH = '/Users/dayana/git repo/machine-learning-class/lab6/knu-2026-machine-learning-final-assignment/'
F = dict(
    tr_acc=PATH+"train-accel.csv", tr_gyr=PATH+"train-gyro.csv", tr_lab=PATH+"train-label.csv",
    te_acc=PATH+"test-accel.csv",  te_gyr=PATH+"test-gyro.csv",  te_lab=PATH+"test-label.csv",
    sub=PATH+"submission.csv",
)
print("config ready")


config ready


In [20]:
train_accel = pd.read_csv(F["tr_acc"]); train_gyro = pd.read_csv(F["tr_gyr"]); train_label = pd.read_csv(F["tr_lab"])
test_accel  = pd.read_csv(F["te_acc"]); test_gyro  = pd.read_csv(F["te_gyr"]); test_label  = pd.read_csv(F["te_lab"])
sub_template = pd.read_csv(F["sub"])

for df in (train_accel, train_gyro, test_accel, test_gyro):
    df.columns = [c.strip().lower() for c in df.columns]

print("accel", train_accel.shape, "gyro", train_gyro.shape, "label", train_label.shape)
print("test_label", test_label.shape, "sub", sub_template.shape)
print("devices(train):", train_accel["device"].value_counts().to_dict())
print("devices(test):",  test_accel["device"].value_counts().to_dict())

accel (2428374, 7) gyro (2433673, 7) label (38015, 4)
test_label (39473, 4) sub (39473, 2)
devices(train): {'samsung': 1558848, 'Apple': 869526}
devices(test): {'samsung': 1564692, 'Apple': 870850, 'unknown': 92768}


In [21]:
SENS = ["x","y","z"]

def fit_device_stats(acc, gyr):
    stats = {}
    for name, df in [("accel", acc), ("gyro", gyr)]:
        g_mean = df[SENS].mean(); g_std = df[SENS].std().replace(0,1)
        per = {}
        for dev, sub in df.groupby("device"):
            per[dev] = (sub[SENS].mean(), sub[SENS].std().replace(0,1))
        stats[name] = dict(global_mean=g_mean, global_std=g_std, per=per)
    return stats

def apply_device_norm(df, st):
    out = df.copy()
    gm, gs, per = st["global_mean"], st["global_std"], st["per"]
    for dev, sub_idx in out.groupby("device").groups.items():
        m, s = per.get(dev, (gm, gs))
        out.loc[sub_idx, SENS] = (out.loc[sub_idx, SENS] - m.values) / s.values
    return out

DEV_STATS = fit_device_stats(train_accel, train_gyro)
train_accel_n = apply_device_norm(train_accel, DEV_STATS["accel"])
train_gyro_n  = apply_device_norm(train_gyro,  DEV_STATS["gyro"])
test_accel_n  = apply_device_norm(test_accel,  DEV_STATS["accel"])
test_gyro_n   = apply_device_norm(test_gyro,   DEV_STATS["gyro"])
print("device-normalized (train stats only)")

device-normalized (train stats only)


In [22]:
# ---- (A) Hampel spike removal + frozen-segment + saturation flags ----
def hampel(x, k=5, nsig=3.0):
    x = np.asarray(x, float)
    if len(x) < 2*k+1: return x
    med = pd.Series(x).rolling(2*k+1, center=True, min_periods=1).median().values
    mad = pd.Series(np.abs(x-med)).rolling(2*k+1, center=True, min_periods=1).median().values
    thr = nsig*1.4826*mad
    out = x.copy(); bad = np.abs(x-med) > thr
    out[bad] = med[bad]
    return out

def clean_and_resample_pid(sub, fs=FS):
    """sub: rows for one pid, sorted by time. Returns uniformly-resampled clean frame."""
    sub = sub.sort_values("time")
    t = sub["time"].values.astype(float)
    if len(t) < 4 or (t.max()-t.min()) <= 0:
        return sub
    # uniform grid
    tg = np.arange(t.min(), t.max(), 1.0/fs)
    res = {"time": tg}
    for c in SENS:
        v = hampel(sub[c].values, k=5, nsig=3.0)
        # frozen-segment guard: tiny jitter to avoid degenerate interp on stalls handled by interp itself
        res[c] = np.interp(tg, t, v)
    out = pd.DataFrame(res)
    for meta in ("pid","direction","device"):
        out[meta] = sub[meta].iloc[0]
    return out

def preprocess_all(acc, gyr):
    a_parts, g_parts = [], []
    for pid in acc["pid"].unique():
        a_parts.append(clean_and_resample_pid(acc[acc.pid==pid]))
    for pid in gyr["pid"].unique():
        g_parts.append(clean_and_resample_pid(gyr[gyr.pid==pid]))
    A = pd.concat(a_parts, ignore_index=True)
    G = pd.concat(g_parts, ignore_index=True)
    return A, G

train_accel_p = train_accel_n.copy(); train_accel_p[SENS] = train_accel_n[SENS]
# resample (this is the big generalization lever)
train_accel_r, train_gyro_r = preprocess_all(train_accel_n, train_gyro_n)
test_accel_r,  test_gyro_r  = preprocess_all(test_accel_n,  test_gyro_n)
print("resampled+cleaned ->", train_accel_r.shape, train_gyro_r.shape, test_accel_r.shape, test_gyro_r.shape)

resampled+cleaned -> (2395969, 7) (2395967, 7) (2536513, 7) (2536520, 7)


In [23]:
def direction_correct(df):
    out = df.copy()
    x, y, z = out["x"].values, out["y"].values, out["z"].values
    d = out["direction"].values
    xb = np.where(np.isin(d,[1,3]),  x, -x)          # lateral
    yb = np.where(np.isin(d,[1,2]),  y, -y)          # forward/back
    zb = z.copy()                                    # vertical
    out["xb"], out["yb"], out["zb"] = xb, yb, zb
    out["magb"] = np.sqrt(xb*xb + yb*yb + zb*zb)
    return out

for nm in ["train_accel_r","train_gyro_r","test_accel_r","test_gyro_r"]:
    globals()[nm] = direction_correct(globals()[nm])
print("direction corrected; cols:", [c for c in train_accel_r.columns if c in ('xb','yb','zb','magb')])

direction corrected; cols: ['xb', 'yb', 'zb', 'magb']


In [24]:
def build_lookup(df):
    df = df.copy()
    df["t_int"] = np.floor(df["time"]).astype(int)
    cols = ["xb","yb","zb","magb"]
    lut = {}
    for (pid, ts), g in df.groupby(["pid","t_int"]):
        lut[(pid, ts)] = g[cols].values.astype(np.float32)
    return lut

LUT = dict(
    tr_acc=build_lookup(train_accel_r), tr_gyr=build_lookup(train_gyro_r),
    te_acc=build_lookup(test_accel_r),  te_gyr=build_lookup(test_gyro_r),
)
print("lookup built:", {k: len(v) for k,v in LUT.items()})

def gather_window(lut, pid, t0, half_s):
    parts = [lut[(pid, t0+dt)] for dt in range(-half_s, half_s+1) if (pid, t0+dt) in lut]
    if not parts: return None
    return np.concatenate(parts, axis=0)   # rows x [xb,yb,zb,magb]

lookup built: {'tr_acc': 47927, 'tr_gyr': 47927, 'te_acc': 50734, 'te_gyr': 50734}


In [25]:
def safe(a, f, d=0.0):
    try:
        v = f(a); return d if not np.isfinite(v) else v
    except Exception: return d

def stat_block(v, pfx):
    f = {}
    f[pfx+"mean"]=safe(v,np.mean); f[pfx+"std"]=safe(v,np.std)
    f[pfx+"min"]=safe(v,np.min);   f[pfx+"max"]=safe(v,np.max)
    f[pfx+"rng"]=f[pfx+"max"]-f[pfx+"min"]
    f[pfx+"med"]=safe(v,np.median)
    f[pfx+"q25"]=safe(v,lambda a:np.percentile(a,25))
    f[pfx+"q75"]=safe(v,lambda a:np.percentile(a,75))
    f[pfx+"iqr"]=f[pfx+"q75"]-f[pfx+"q25"]
    f[pfx+"rms"]=safe(v,lambda a:np.sqrt(np.mean(a*a)))
    f[pfx+"mad"]=safe(v,lambda a:np.mean(np.abs(a-np.mean(a))))
    f[pfx+"skew"]=safe(v,lambda a:((a-a.mean())**3).mean()/(a.std()**3+1e-9))
    f[pfx+"kurt"]=safe(v,lambda a:((a-a.mean())**4).mean()/(a.std()**4+1e-9))
    f[pfx+"energy"]=safe(v,lambda a:np.sum(a*a)/len(a))
    f[pfx+"zcr"]=safe(v,lambda a:np.mean(np.abs(np.diff(np.sign(a-a.mean())))>0))
    return f

def fft_block(v, pfx, fs=FS):
    v = v - np.mean(v); n=len(v)
    if n < 8: return {pfx+k:0.0 for k in ["domf","domp","ent","cent","flat","p_low","p_high"]}
    Y = np.abs(rfft(v*np.hanning(n))); fr = rfftfreq(n, 1/fs)
    Y[0]=0; P = Y*Y; tot=P.sum()+1e-9
    i = int(np.argmax(P))
    # parabolic interpolation for sub-bin dominant frequency (sharper cadence)
    if 0 < i < len(P)-1:
        a,b,c = P[i-1],P[i],P[i+1]; denom=(a-2*b+c)
        d = 0.5*(a-c)/denom if denom!=0 else 0.0
    else: d=0.0
    domf = (i+d)*fs/n
    pr = P/tot
    ent = -np.sum(pr*np.log(pr+1e-12))
    cent = np.sum(fr*P)/tot
    flat = np.exp(np.mean(np.log(P+1e-12)))/(np.mean(P)+1e-12)  # spectral flatness
    p_low  = P[(fr>=0.5)&(fr<2.0)].sum()/tot
    p_high = P[(fr>=2.0)&(fr<5.0)].sum()/tot
    return {pfx+"domf":domf, pfx+"domp":P[i]/tot, pfx+"ent":ent,
            pfx+"cent":cent, pfx+"flat":flat, pfx+"p_low":p_low, pfx+"p_high":p_high}

def autocorr_period(v, fs=FS):
    v = v-np.mean(v); n=len(v)
    if n<16 or np.std(v)<1e-6: return 0.0,0.0
    ac = np.correlate(v,v,"full")[n-1:]; ac/= (ac[0]+1e-9)
    lo,hi = int(fs*0.2), min(int(fs*2.0), n-1)
    if hi<=lo: return 0.0,0.0
    seg = ac[lo:hi]; k=int(np.argmax(seg))+lo
    return fs/k if k>0 else 0.0, float(ac[k])

In [26]:
def p6_extra(acc_w, gyr_w, fs=FS):
    """acc_w/gyr_w: rows x [xb,yb,zb,magb]. Returns dict of NEW features."""
    f = {}
    if acc_w is None or len(acc_w) < 8:
        return {f"p6_{k}":0.0 for k in
                ["sym_lat","vert_lat_ratio","vert_energy","lat_energy",
                 "ag_coupling","still_frac","spec_flat_mag","domf_mag",
                 "pca_vert","jerk_rms"]}
    xb,yb,zb,mag = acc_w[:,0],acc_w[:,1],acc_w[:,2],acc_w[:,3]

    # symmetry: lunges asymmetric vs squats symmetric (skew of lateral)
    f["p6_sym_lat"] = safe(xb, lambda a: abs(((a-a.mean())**3).mean())/(a.std()**3+1e-9))

    # vertical vs lateral energy (squat=vertical, mtnclimb=lateral/mixed)
    ve = np.sum(zb*zb)/len(zb); le = np.sum(xb*xb)/len(xb)
    f["p6_vert_energy"]=ve; f["p6_lat_energy"]=le
    f["p6_vert_lat_ratio"]= ve/(le+1e-6)

    # sharpened cadence on magnitude
    fb = fft_block(mag,"m_",fs)
    f["p6_domf_mag"]=fb["m_domf"]; f["p6_spec_flat_mag"]=fb["m_flat"]

    # stillness for Rest: fraction of low-energy samples + jerk
    jerk = np.diff(mag)*fs
    f["p6_jerk_rms"]= safe(jerk, lambda a: np.sqrt(np.mean(a*a)))
    thr = 0.05*(np.std(mag)+1e-6)
    f["p6_still_frac"]= float(np.mean(np.abs(mag-np.mean(mag))<thr))

    # PCA principal-axis verticality
    M = np.vstack([xb,yb,zb]); M = M-M.mean(1,keepdims=True)
    try:
        u,s,vt = np.linalg.svd(M, full_matrices=False)
        f["p6_pca_vert"]= abs(u[2,0])  # vertical loading of 1st PC
    except Exception:
        f["p6_pca_vert"]=0.0

    # accel-gyro coupling (rotational exercises couple linear+angular)
    if gyr_w is not None and len(gyr_w)>=8:
        gm = gyr_w[:,3]; n=min(len(mag),len(gm))
        a1=mag[:n]-mag[:n].mean(); g1=gm[:n]-gm[:n].mean()
        denom=(np.std(a1)*np.std(g1)*n+1e-9)
        f["p6_ag_coupling"]= float(np.dot(a1,g1)/denom)
    else:
        f["p6_ag_coupling"]=0.0
    return f

In [27]:
CH = {0:"xb",1:"yb",2:"zb",3:"magb"}

def base_features(acc_w, gyr_w):
    f={}
    for src,W in [("a",acc_w),("g",gyr_w)]:
        if W is None: 
            continue
        for idx,nm in CH.items():
            col=W[:,idx]
            f.update(stat_block(col, f"{src}_{nm}_s_"))
            f.update(fft_block(col,  f"{src}_{nm}_f_"))
            p,c = autocorr_period(col)
            f[f"{src}_{nm}_acp"]=p; f[f"{src}_{nm}_acc"]=c
    return f

def neighbor_energy(lut_a, pid, t0):
    out={}
    for tag,dt in [("prev",-2),("next",2)]:
        w = lut_a.get((pid,t0+dt))
        out[f"nb_{tag}_e"]= float(np.mean(w[:,3]**2)) if w is not None else 0.0
    return out

def make_row(la, lg, pid, t0):
    a_short=gather_window(la,pid,t0,SHORT_S); 
    a_long =gather_window(la,pid,t0,LONG_S)
    g_long =gather_window(lg,pid,t0,LONG_S)
    a_ext  =gather_window(la,pid,t0,EXT_S)
    f={}
    f.update(base_features(a_long, g_long))
    if a_short is not None: f.update(stat_block(a_short[:,3],"sh_mag_"))
    if a_ext is not None:   f.update(fft_block(a_ext[:,3],"ext_mag_"))
    f.update(p6_extra(a_long, g_long))
    f.update(neighbor_energy(la,pid,t0))
    return f

def build_matrix(label_df, la, lg):
    rows=[]; idx_keep=[]
    for r in label_df.itertuples():
        pid=r.pid; t0=int(np.floor(r.time))
        if (pid,t0) not in la and (pid,t0) not in lg:
            rows.append(None); continue
        rows.append(make_row(la,lg,pid,t0)); idx_keep.append(r.Index)
    feat_rows=[x for x in rows if x is not None]
    X=pd.DataFrame(feat_rows).fillna(0.0)
    return X, idx_keep

X_tr, keep_tr = build_matrix(train_label, LUT["tr_acc"], LUT["tr_gyr"])
X_te, keep_te = build_matrix(test_label,  LUT["te_acc"], LUT["te_gyr"])
X_tr, X_te = X_tr.align(X_te, join="outer", axis=1, fill_value=0.0)
y_tr = train_label.loc[keep_tr,"workout"].values
g_tr = train_label.loc[keep_tr,"pid"].values
print("X_tr",X_tr.shape,"X_te",X_te.shape,"classes",np.unique(y_tr))

X_tr (37821, 226) X_te (38349, 226) classes [0 1 2 3 4 5 6 7 8]


In [28]:
N_FOLDS=5
gkf=GroupKFold(n_splits=N_FOLDS)
oof_lgbm=np.zeros((len(X_tr),N_CLASS)); oof_cat=np.zeros((len(X_tr),N_CLASS))
te_lgbm=np.zeros((len(X_te),N_CLASS));  te_cat=np.zeros((len(X_te),N_CLASS))

lgb_par=dict(objective="multiclass",num_class=N_CLASS,learning_rate=0.03,
             num_leaves=64,feature_fraction=0.7,bagging_fraction=0.8,bagging_freq=1,
             min_child_samples=40,n_estimators=1200,class_weight="balanced",
             random_state=SEED,verbose=-1)

for k,(tr,va) in enumerate(gkf.split(X_tr,y_tr,g_tr)):
    m=lgb.LGBMClassifier(**lgb_par)
    m.fit(X_tr.iloc[tr],y_tr[tr],eval_set=[(X_tr.iloc[va],y_tr[va])],
          callbacks=[lgb.early_stopping(80,verbose=False)])
    oof_lgbm[va]=m.predict_proba(X_tr.iloc[va]); te_lgbm+=m.predict_proba(X_te)/N_FOLDS

    c=CatBoostClassifier(iterations=1500,learning_rate=0.03,depth=7,
        loss_function="MultiClass",auto_class_weights="Balanced",
        random_seed=SEED,verbose=0)
    c.fit(X_tr.iloc[tr],y_tr[tr],eval_set=(X_tr.iloc[va],y_tr[va]),
          early_stopping_rounds=80)
    oof_cat[va]=c.predict_proba(X_tr.iloc[va]); te_cat+=c.predict_proba(X_te)/N_FOLDS
    print(f"fold{k} done")
print("stage-2 OOF ready")

fold0 done
fold1 done
fold2 done
fold3 done
fold4 done
stage-2 OOF ready


In [29]:
y_active=(y_tr!=8).astype(int)
oof_is_active=np.zeros(len(X_tr)); te_is_active=np.zeros(len(X_te))
lgb_b=dict(objective="binary",learning_rate=0.03,num_leaves=48,
           feature_fraction=0.7,bagging_fraction=0.8,bagging_freq=1,
           min_child_samples=40,n_estimators=1000,scale_pos_weight=8.0,
           random_state=SEED,verbose=-1)
for k,(tr,va) in enumerate(gkf.split(X_tr,y_active,g_tr)):
    m=lgb.LGBMClassifier(**lgb_b)
    m.fit(X_tr.iloc[tr],y_active[tr],eval_set=[(X_tr.iloc[va],y_active[va])],
          callbacks=[lgb.early_stopping(80,verbose=False)])
    oof_is_active[va]=m.predict_proba(X_tr.iloc[va])[:,1]
    te_is_active+=m.predict_proba(X_te)[:,1]/N_FOLDS
print("stage-1 OOF ready")

stage-1 OOF ready


In [30]:
def soft_prob_smooth(P, label_df, keep_idx, win=2, alpha=0.6):
    """temporal smoothing within (pid) ordered by time; leakage-safe (uses probs only)."""
    out=P.copy()
    df=label_df.loc[keep_idx,["pid","time"]].reset_index(drop=True)
    for pid in df.pid.unique():
        ix=np.where(df.pid.values==pid)[0]
        order=ix[np.argsort(df.time.values[ix])]
        sm=P[order].copy()
        for i in range(len(order)):
            lo=max(0,i-win); hi=min(len(order),i+win+1)
            sm[i]=alpha*P[order[i]]+(1-alpha)*P[order[lo:hi]].mean(0)
        out[order]=sm
    return out

def apply_gate(P, is_active, thr):
    Q=P.copy()
    rest=is_active<thr
    Q[rest]=0; Q[rest,8]=1.0
    act=~rest
    Q[act,8]=0; Q[act]/=Q[act].sum(1,keepdims=True)+1e-9
    return Q

best=(-1,None)
for w in [0.40,0.45,0.50,0.55]:
    P=w*oof_lgbm+(1-w)*oof_cat
    Ps=soft_prob_smooth(P,train_label,keep_tr)
    for thr in [0.55,0.60,0.65,0.70,0.74]:
        Q=apply_gate(Ps,oof_is_active,thr)
        ba=balanced_accuracy_score(y_tr,Q.argmax(1))
        if ba>best[0]: best=(ba,(w,thr))
oof_ba_p6=best[0]; W,THR=best[1]
print(f"p6 OOF BA={oof_ba_p6:.5f}  (w={W}, thr={THR})  | p5 baseline={P5_BASELINE_BA}")

from sklearn.metrics import classification_report
P=W*oof_lgbm+(1-W)*oof_cat
Q=apply_gate(soft_prob_smooth(P,train_label,keep_tr),oof_is_active,THR)
print(classification_report(y_tr,Q.argmax(1),digits=3))

p6 OOF BA=0.90868  (w=0.45, thr=0.65)  | p5 baseline=0.91879
              precision    recall  f1-score   support

           0      0.941     0.952     0.947      3389
           1      0.928     0.911     0.920      3523
           2      0.916     0.877     0.896      3575
           3      0.923     0.897     0.909      3697
           4      0.889     0.943     0.915      3610
           5      0.899     0.934     0.916      3627
           6      0.847     0.885     0.865      3589
           7      0.881     0.876     0.878      3601
           8      0.926     0.903     0.914      9210

    accuracy                          0.908     37821
   macro avg      0.905     0.909     0.907     37821
weighted avg      0.908     0.908     0.908     37821



In [32]:
# Test-side prediction with chosen config
P_te=W*te_lgbm+(1-W)*te_cat
P_te_s=soft_prob_smooth(P_te,test_label,keep_te)
Q_te=apply_gate(P_te_s,te_is_active,THR)
pred_te=Q_te.argmax(1)

# map predictions back to ids
test_pred = pd.Series(8, index=test_label.index)        # default Rest (covers no-sensor pids, e.g. pid71)
test_pred.loc[keep_te] = pred_te
sub = sub_template.copy()
id2lab = dict(zip(test_label["id"], test_pred.values))
sub["workout"] = sub["id"].map(id2lab).fillna(8).astype(int)

# ---- always write + always print result ----
sub.to_csv("submission_p6.csv", index=False)

delta = oof_ba_p6 - P5_BASELINE_BA
status = "✅ IMPROVED" if delta >= 0 else "⚠️ REGRESSION"

print("="*55)
print(f"{status}")
print(f"p6 OOF BA   : {oof_ba_p6:.5f}")
print(f"p5 baseline : {P5_BASELINE_BA:.5f}")
print(f"delta       : {delta:+.5f}")
print(f"config      : w={W}, thr={THR}")
print(f"file        : submission_p6.csv written ({sub.shape[0]} rows)")
print("pred dist   :", pd.Series(sub['workout']).value_counts().sort_index().to_dict())
print("="*55)
if delta < 0:
    print("Note: p6 is below p5 baseline. Inspect per-class report; "
          "consider disabling weakest new feature block (neighbor_energy or p6_extra) and re-run.")

⚠️ REGRESSION
p6 OOF BA   : 0.90868
p5 baseline : 0.91879
delta       : -0.01011
config      : w=0.45, thr=0.65
file        : submission_p6.csv written (39473 rows)
pred dist   : {0: 3776, 1: 3523, 2: 3864, 3: 3539, 4: 4011, 5: 3827, 6: 4530, 7: 3763, 8: 8640}
Note: p6 is below p5 baseline. Inspect per-class report; consider disabling weakest new feature block (neighbor_energy or p6_extra) and re-run.
